In [0]:
display(
spark.sql('''
          select * from sdp_practice.source.customer_raw
          ''')
)

In [0]:
%sql
create table if not exists  sdp_practice.source.sales
(
    order_id int,
    product_id int,
    revenue float,
    date DATE,
    store_id int
);

insert into sdp_practice.source.sales
values 
-- (1, 100, 100.00, '2022-01-01', 1),
-- (2, 101, 200.00, '2022-01-02', 2),
-- (3, 102, 300.00, '2022-01-03', 3),
-- (4, 103, 400.00, '2022-01-04', 4);
(7, 100, 100.00, '2022-01-01', 1),
(2, 101, 200.00, '2022-01-02', 2);


In [0]:
%sql
select * from sdp_practice.source.sales;

In [0]:
%sql
select * from sdp_practice.target.cur_sales_stream;

In [0]:
%sql
DESCRIBE HISTORY sdp_practice.target.enr_sales_stream;

In [0]:
%sql
select * from sdp_practice.source.sales_north;

In [0]:
%sql
delete from sdp_practice.source.sales_north
where product_id in
(
select product_id 
from
(
select *,
    dense_rank() over(partition by order_id order by product_id asc) as rank
from sdp_practice.source.sales_north
) 
where rank > 1
)

In [0]:
%sql
select * from sdp_practice.source.sales_south;

In [0]:
%sql
select * from sdp_practice.target_sdp.tot_sales

In [0]:
%sql
select * from sdp_practice.source.products;

#### **Slowly Changing Dimension**

In [0]:
%sql
drop table sdp_practice.source.products;

In [0]:
%sql
create table if not exists sdp_practice.source.products
(
    product_id int,
    product_name string,
    category string,
    subcategory string,
    updated_at timestamp
);


insert into sdp_practice.source.products
values
(1001, 'Product1', 'Category1', 'Subcategory1', current_timestamp()),
(1002, 'Product2', 'Category2', 'Subcategory2', current_timestamp()),
(1003, 'Product3', 'Category3', 'Subcategory3', current_timestamp()),
(1004, 'Product4', 'Category4', 'Subcategory4', current_timestamp());


select * from sdp_practice.source.products;

In [0]:
%sql
select * from sdp_practice.target_sdp.products_scd2
order by product_id,subcategory, `__END_AT` desc

In [0]:
%sql
select * from sdp_practice.target_sdp.products_scd1;

###Insert product 3 with new timestamp

In [0]:
%sql
insert into sdp_practice.source.products
values
(NULL, 'Product2', 'Category4', 'Subcategory4', current_timestamp())

In [0]:
%sql
select * from sdp_practice.source.products;

In [0]:
%sql
select * from sdp_practice.target_sdp.products_table